# V2: 최고 성능 구조 안정성 예측 모델

v1 대비 주요 개선:
- SAM Optimizer (Sharpness-Aware Minimization)
- SWA (Stochastic Weight Averaging)
- **EMA (Exponential Moving Average)**
- Progressive Resizing (224→384, 2-phase)
- GeM Pooling
- **SE Fusion + LayerNorm Head**
- **Synchronized Augmentation (front/top 동기화)**
- 강화된 Augmentation (CoarseDropout, CLAHE)
- Focal Loss 옵션
- 고급 확률 보정 (Temp/Platt/Isotonic 자동 선택)
- **Multi-Scale TTA (3 scales × 2 flips)**
- Pseudo-Labeling
- Stacking Ensemble (Level 2 Meta-Learner)
- **6종 Backbone 사전 설정 (BACKBONE_CONFIGS)**

**실행 순서:** backbone별로 `Config.from_backbone('key')` 로 Config 생성 → Run All 반복 (6회) → 마지막에 앙상블+stacking

## Section 0: Imports + Config

In [1]:
import copy
import os
import random
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional

import albumentations as A
import cv2
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from albumentations.pytorch import ToTensorV2
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from torch.amp import autocast
from torch.optim.swa_utils import AveragedModel, update_bn
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

warnings.filterwarnings('ignore')


def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

Device: cuda
GPU: NVIDIA GeForce RTX 3080
VRAM: 10.0 GB


In [2]:
# === Backbone Configs (batch_size tuned for RTX 3080 10GB) ===
BACKBONE_CONFIGS = {
    'convnextv2_base': {
        'name': 'convnextv2_base.fcmae_ft_in22k_in1k_384',
        'img_size': 384,
        'batch_size': 2,
    },
    'eva02_base': {
        'name': 'eva02_base_patch14_448.mim_in22k_ft_in22k_in1k',
        'img_size': 448,
        'batch_size': 2,
    },
    'swinv2_base': {
        'name': 'swinv2_base_window12to24_192to384.ms_in22k_ft_in1k',
        'img_size': 384,
        'batch_size': 2,
    },
    'maxvit_base': {
        'name': 'maxvit_base_tf_384.in21k_ft_in1k',
        'img_size': 384,
        'batch_size': 2,
    },
    'efficientnetv2_m': {
        'name': 'tf_efficientnetv2_m.in21k_ft_in1k',
        'img_size': 384,
        'batch_size': 4,
    },
    'caformer_b36': {
        'name': 'caformer_b36.sail_in22k_ft_in1k_384',
        'img_size': 384,
        'batch_size': 2,
    },
}


@dataclass
class Config:
    # === backbone 교체 시 여기만 수정 ===
    backbone: str = 'convnextv2_base.fcmae_ft_in22k_in1k_384'
    exp_name: str = 'v2_convnextv2_base'
    img_size: int = 384

    # Training (3080 10GB: bs↓ grad_accum↑ → effective batch 동일)
    epochs: int = 30
    batch_size: int = 2
    grad_accum: int = 16  # effective batch = 2×16 = 32
    lr_backbone: float = 1e-4
    lr_head: float = 1e-3
    weight_decay: float = 0.01
    warmup_epochs: int = 2
    early_stopping_patience: int = 7

    # SAM Optimizer
    use_sam: bool = True
    sam_rho: float = 0.05

    # SWA (후반 25%)
    use_swa: bool = True
    swa_start_pct: float = 0.75
    swa_lr: float = 1e-5

    # EMA (CPU에 보관 → VRAM 절약)
    use_ema: bool = True
    ema_decay: float = 0.9995

    # Progressive Resizing
    img_size_phase1: int = 224
    progressive_resize: bool = True
    phase1_pct: float = 0.4

    # Regularization
    label_smoothing: float = 0.05
    mixup_alpha: float = 0.3
    cutmix_alpha: float = 1.0
    mix_prob: float = 0.5
    drop_path_rate: float = 0.2

    # Knowledge Distillation
    use_kd: bool = True
    kd_alpha: float = 0.3
    kd_temperature: float = 3.0

    # Focal Loss
    use_focal: bool = False
    focal_gamma: float = 2.0

    # GeM Pooling
    use_gem: bool = True
    gem_p_init: float = 3.0

    # Fold
    n_folds: int = 5
    seed: int = 42

    # TTA
    tta_count: int = 7
    tta_scales: Optional[list] = None

    # Pseudo-Labeling
    use_pseudo: bool = True
    pseudo_threshold: float = 0.95

    # Paths
    data_dir: str = '../data'
    output_dir: str = '../outputs'

    def __post_init__(self):
        if self.tta_scales is None:
            self.tta_scales = [self.img_size, self.img_size + 64, self.img_size + 128]

    @classmethod
    def from_backbone(cls, key: str, **overrides):
        """BACKBONE_CONFIGS에서 backbone key로 Config 생성."""
        bc = BACKBONE_CONFIGS[key]
        defaults = {
            'backbone': bc['name'],
            'exp_name': f'v2_{key}',
            'img_size': bc['img_size'],
            'batch_size': bc.get('batch_size', 2),
        }
        defaults.update(overrides)
        return cls(**defaults)


cfg = Config()
seed_everything(cfg.seed)

exp_dir = Path(cfg.output_dir) / cfg.exp_name
exp_dir.mkdir(parents=True, exist_ok=True)
print(f'Experiment: {cfg.exp_name}')
print(f'Backbone: {cfg.backbone}')
print(f'Image size: {cfg.img_size} (Phase1: {cfg.img_size_phase1})')
print(f'Batch: {cfg.batch_size} × grad_accum {cfg.grad_accum} = effective {cfg.batch_size * cfg.grad_accum}')
print(f'SAM: {cfg.use_sam} | SWA: {cfg.use_swa} | GeM: {cfg.use_gem} | EMA: {cfg.use_ema}')
print(f'Progressive resize: {cfg.progressive_resize}')
print(f'Output: {exp_dir}')

Experiment: v2_convnextv2_base
Backbone: convnextv2_base.fcmae_ft_in22k_in1k_384
Image size: 384 (Phase1: 224)
Batch: 2 × grad_accum 16 = effective 32
SAM: True | SWA: True | GeM: True | EMA: True
Progressive resize: True
Output: ..\outputs\v2_convnextv2_base


## Section 1: Dataset + 강화 Augmentation

In [3]:
# 데이터 로딩
data_dir = Path(cfg.data_dir)
train_df = pd.read_csv(data_dir / 'train.csv')
dev_df = pd.read_csv(data_dir / 'dev.csv')
test_df = pd.read_csv(data_dir / 'sample_submission.csv')

# Train + Dev 통합
train_df['split'] = 'train'
dev_df['split'] = 'dev'
all_df = pd.concat([train_df, dev_df], ignore_index=True)
all_df['label_int'] = (all_df['label'] == 'unstable').astype(int)

# Soft label 로딩
soft_labels_path = data_dir / 'soft_labels.csv'
if soft_labels_path.exists():
    soft_df = pd.read_csv(soft_labels_path)
    all_df = all_df.merge(soft_df[['id', 'soft_unstable_prob']], on='id', how='left')
    all_df['soft_unstable_prob'] = all_df['soft_unstable_prob'].fillna(
        all_df['label_int'].astype(float)
    )
    print(f'Soft labels loaded: {soft_df.shape[0]} samples')
else:
    all_df['soft_unstable_prob'] = all_df['label_int'].astype(float)
    print('WARNING: soft_labels.csv not found, using hard labels')

print(f'Total samples: {len(all_df)} (train: {len(train_df)}, dev: {len(dev_df)})')
print(f'Label distribution: {all_df["label"].value_counts().to_dict()}')

Soft labels loaded: 1000 samples
Total samples: 1100 (train: 1000, dev: 100)
Label distribution: {'unstable': 552, 'stable': 548}


In [4]:
def get_train_transforms(img_size):
    """v2 강화 augmentation: +CoarseDropout, +CLAHE, synchronized front/top"""
    return A.Compose([
        A.Resize(img_size, img_size),

        # CLAHE: 조명 균일화
        A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.3),

        # 색상 변환
        A.RandomBrightnessContrast(
            brightness_limit=(-0.3, 0.1),
            contrast_limit=(-0.3, 0.3),
            p=0.8
        ),
        A.ColorJitter(
            brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1, p=0.6
        ),
        A.RandomGamma(gamma_limit=(70, 130), p=0.4),
        A.RandomToneCurve(scale=0.2, p=0.3),

        # 카메라 각도 변형
        A.Perspective(scale=(0.02, 0.08), p=0.5),
        A.Affine(
            scale=(0.85, 1.15),
            translate_percent=(-0.1, 0.1),
            rotate=(-15, 15),
            shear=(-10, 10),
            p=0.6
        ),

        # 기본 변형
        A.HorizontalFlip(p=0.5),
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
        A.GaussNoise(std_range=(0.02, 0.08), p=0.2),

        # CoarseDropout
        A.CoarseDropout(
            num_holes_range=(4, 8),
            hole_height_range=(int(img_size * 0.05), int(img_size * 0.15)),
            hole_width_range=(int(img_size * 0.05), int(img_size * 0.15)),
            fill=0,
            p=0.3
        ),

        # 정규화
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ], additional_targets={'top': 'image'})


def get_val_transforms(img_size):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ], additional_targets={'top': 'image'})


def get_tta_transforms(img_size):
    """v2 강화 TTA: 더 다양한 변형"""
    return A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ], additional_targets={'top': 'image'})


def get_multiscale_tta_transforms(img_size, flip=False):
    """Multi-scale TTA: 특정 크기 + optional flip"""
    transforms_list = [
        A.Resize(img_size, img_size),
    ]
    if flip:
        transforms_list.append(A.HorizontalFlip(p=1.0))
    transforms_list.extend([
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])
    return A.Compose(transforms_list, additional_targets={'top': 'image'})

In [5]:
class StructuralDataset(Dataset):
    def __init__(self, df, data_dir, transforms=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.data_dir = Path(data_dir)
        self.transforms = transforms
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        sample_id = row['id']

        split = row.get("split", "train")
        if self.is_test or split == "test":
            base = self.data_dir / "test" / sample_id
        elif split == "dev":
            base = self.data_dir / "dev" / sample_id
        else:
            base = self.data_dir / "train" / sample_id

        front = cv2.imread(str(base / 'front.png'))
        front = cv2.cvtColor(front, cv2.COLOR_BGR2RGB)
        top = cv2.imread(str(base / 'top.png'))
        top = cv2.cvtColor(top, cv2.COLOR_BGR2RGB)

        if self.transforms:
            # Synchronized augmentation: front/top에 동일 random transform 적용
            augmented = self.transforms(image=front, top=top)
            front = augmented['image']
            top = augmented['top']

        result = {'front': front, 'top': top, 'id': sample_id}

        if not self.is_test:
            result['label'] = int(row['label_int'])
            result['soft_label'] = float(row.get('soft_unstable_prob', row['label_int']))

        return result


# Quick test
test_ds = StructuralDataset(
    all_df.head(2), data_dir, get_train_transforms(cfg.img_size)
)
sample = test_ds[0]
print(f'Front shape: {sample["front"].shape}')
print(f'Top shape: {sample["top"].shape}')
print(f'Label: {sample["label"]}, Soft: {sample["soft_label"]:.4f}')

# Synchronized augmentation 검증
print(f'Front/Top 동일 변환 적용 확인: shapes match = {sample["front"].shape == sample["top"].shape}')

Front shape: torch.Size([3, 384, 384])
Top shape: torch.Size([3, 384, 384])
Label: 1, Soft: 0.9993
Front/Top 동일 변환 적용 확인: shapes match = True


## Section 2: Model (DualStreamModel + GeM Pooling)

In [6]:
class GeM(nn.Module):
    """Generalized Mean Pooling with learnable p parameter."""
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return x.clamp(min=self.eps).pow(self.p).mean(dim=[-2, -1]).pow(1.0 / self.p)


class SEFusion(nn.Module):
    """Squeeze-Excitation based feature fusion for dual-stream concat."""
    def __init__(self, in_features, reduction=16):
        super().__init__()
        mid = max(in_features // reduction, 32)
        self.fc = nn.Sequential(
            nn.Linear(in_features, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, in_features, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        # x: (B, 2*feat_dim) — concatenated front+top features
        scale = self.fc(x)
        return x * scale


class DualStreamModelV2(nn.Module):
    def __init__(self, backbone_name, num_classes=2, drop_path_rate=0.15,
                 use_gem=True, gem_p=3.0):
        super().__init__()

        self.backbone_front = timm.create_model(
            backbone_name, pretrained=True, num_classes=0,
            drop_path_rate=drop_path_rate
        )
        self.backbone_top = timm.create_model(
            backbone_name, pretrained=True, num_classes=0,
            drop_path_rate=drop_path_rate
        )

        # Gradient checkpointing
        if hasattr(self.backbone_front, 'set_grad_checkpointing'):
            self.backbone_front.set_grad_checkpointing(True)
            self.backbone_top.set_grad_checkpointing(True)

        feat_dim = self.backbone_front.num_features

        # GeM Pooling
        self.use_gem = use_gem
        if use_gem:
            self.gem_front = GeM(p=gem_p)
            self.gem_top = GeM(p=gem_p)

        # SE Fusion
        self.fusion = SEFusion(feat_dim * 2)

        # Head with LayerNorm
        self.head = nn.Sequential(
            nn.LayerNorm(feat_dim * 2),
            nn.Linear(feat_dim * 2, 512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.LayerNorm(512),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes),
        )
        self.num_classes = num_classes
        self.feat_dim = feat_dim

    def _forward_features(self, backbone, gem, x):
        """backbone의 feature map을 추출하고 GeM pooling 적용"""
        if self.use_gem:
            feat_map = backbone.forward_features(x)
            if feat_map.dim() == 3:
                # ViT-like: (B, N, C)
                if hasattr(backbone, 'num_prefix_tokens'):
                    feat_map = feat_map[:, backbone.num_prefix_tokens:, :]
                B, N, C = feat_map.shape
                H = W = int(N ** 0.5)
                if H * W == N:
                    feat_map = feat_map.permute(0, 2, 1).reshape(B, C, H, W)
                    feat = gem(feat_map)
                else:
                    feat = feat_map.mean(dim=1)
            elif feat_map.dim() == 4:
                feat = gem(feat_map)
            else:
                feat = feat_map
        else:
            feat = backbone(x)
        return feat

    def forward(self, front, top):
        if self.use_gem:
            feat_front = self._forward_features(
                self.backbone_front, self.gem_front, front
            )
            feat_top = self._forward_features(
                self.backbone_top, self.gem_top, top
            )
        else:
            feat_front = self.backbone_front(front)
            feat_top = self.backbone_top(top)

        combined = torch.cat([feat_front, feat_top], dim=1)
        combined = self.fusion(combined)
        logits = self.head(combined)
        return logits


# 모델 생성 테스트
model = DualStreamModelV2(
    cfg.backbone, drop_path_rate=cfg.drop_path_rate,
    use_gem=cfg.use_gem, gem_p=cfg.gem_p_init
)
print(f'Feature dim: {model.feat_dim}')
total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Total params: {total_params:.1f}M')

# Forward test
with torch.no_grad():
    dummy_f = torch.randn(2, 3, cfg.img_size, cfg.img_size)
    dummy_t = torch.randn(2, 3, cfg.img_size, cfg.img_size)
    out = model(dummy_f, dummy_t)
    print(f'Output shape: {out.shape}')

del model
torch.cuda.empty_cache()

Feature dim: 1024
Total params: 177.0M
Output shape: torch.Size([2, 2])


## Section 3: SAM Optimizer + Loss + Scheduler

In [7]:
class SAM(torch.optim.Optimizer):
    """Sharpness-Aware Minimization.
    https://arxiv.org/abs/2010.01412
    """
    def __init__(self, params, base_optimizer, rho=0.05, **kwargs):
        defaults = dict(rho=rho, **kwargs)
        super().__init__(params, defaults)
        self.base_optimizer = base_optimizer(self.param_groups, **kwargs)
        self.param_groups = self.base_optimizer.param_groups
        self._sam_state = {}

    @torch.no_grad()
    def first_step(self):
        """Ascent step: move to worst-case point."""
        grad_norm = self._grad_norm()
        for group in self.param_groups:
            scale = group['rho'] / (grad_norm + 1e-12)
            for p in group['params']:
                if p.grad is None:
                    continue
                e_w = p.grad * scale
                p.add_(e_w)
                self._sam_state[p] = e_w

    @torch.no_grad()
    def second_step(self):
        """Descent step: optimize at worst-case point, then restore."""
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                p.sub_(self._sam_state[p])
        self.base_optimizer.step()

    @torch.no_grad()
    def step(self, closure=None):
        """Fallback for non-SAM usage."""
        self.base_optimizer.step(closure)

    def _grad_norm(self):
        shared_device = self.param_groups[0]['params'][0].device
        norm = torch.norm(
            torch.stack([
                p.grad.norm(p=2).to(shared_device)
                for group in self.param_groups
                for p in group['params']
                if p.grad is not None
            ]),
            p=2
        )
        return norm

    def zero_grad(self, set_to_none=False):
        self.base_optimizer.zero_grad(set_to_none=set_to_none)

In [8]:
class ModelEmaV2(nn.Module):
    """Exponential Moving Average of model weights.
    CPU에 shadow copy를 유지하여 VRAM 절약.
    """
    def __init__(self, model, decay=0.9995):
        super().__init__()
        self.module = copy.deepcopy(model).cpu()  # CPU에 보관
        self.module.eval()
        self.decay = decay

    @torch.no_grad()
    def update(self, model):
        for ema_p, model_p in zip(self.module.parameters(), model.parameters()):
            ema_p.data.mul_(self.decay).add_(model_p.data.cpu(), alpha=1.0 - self.decay)

    def forward(self, *args, **kwargs):
        return self.module(*args, **kwargs)


class FocalLoss(nn.Module):
    """Focal Loss for class imbalance + hard example mining."""
    def __init__(self, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        num_classes = logits.size(1)
        if self.label_smoothing > 0:
            targets_onehot = F.one_hot(targets, num_classes).float()
            targets_smooth = (
                targets_onehot * (1 - self.label_smoothing)
                + self.label_smoothing / num_classes
            )
        else:
            targets_smooth = F.one_hot(targets, num_classes).float()

        log_probs = F.log_softmax(logits, dim=1)
        probs = torch.exp(log_probs)

        focal_weight = (1 - probs) ** self.gamma
        loss = -focal_weight * targets_smooth * log_probs
        return loss.sum(dim=1).mean()


def mixup_data(front, top, labels, soft_labels, alpha=0.3):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    batch_size = front.size(0)
    index = torch.randperm(batch_size).to(front.device)
    mixed_front = lam * front + (1 - lam) * front[index]
    mixed_top = lam * top + (1 - lam) * top[index]
    return mixed_front, mixed_top, labels, labels[index], soft_labels, soft_labels[index], lam


def cutmix_data(front, top, labels, soft_labels, alpha=1.0):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    batch_size = front.size(0)
    index = torch.randperm(batch_size).to(front.device)
    _, _, H, W = front.shape
    cut_rat = np.sqrt(1.0 - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1 = np.clip(cx - cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y2 = np.clip(cy + cut_h // 2, 0, H)

    mixed_front = front.clone()
    mixed_front[:, :, y1:y2, x1:x2] = front[index, :, y1:y2, x1:x2]
    mixed_top = top.clone()
    mixed_top[:, :, y1:y2, x1:x2] = top[index, :, y1:y2, x1:x2]
    lam = 1 - ((x2 - x1) * (y2 - y1) / (W * H))
    return mixed_front, mixed_top, labels, labels[index], soft_labels, soft_labels[index], lam


def compute_loss(logits, labels_a, labels_b, soft_a, soft_b, lam, cfg):
    """KD + Label Smoothing + Mixup/CutMix + optional Focal Loss"""
    if cfg.use_focal:
        loss_fn = FocalLoss(gamma=cfg.focal_gamma, label_smoothing=cfg.label_smoothing)
    else:
        loss_fn = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)

    hard_loss = lam * loss_fn(logits, labels_a) + (1 - lam) * loss_fn(logits, labels_b)

    if cfg.use_kd:
        T = cfg.kd_temperature
        soft_targets_a = torch.stack([1 - soft_a, soft_a], dim=1)
        soft_targets_b = torch.stack([1 - soft_b, soft_b], dim=1)
        soft_targets = lam * soft_targets_a + (1 - lam) * soft_targets_b
        log_probs = F.log_softmax(logits / T, dim=1)
        soft_targets_scaled = F.softmax(soft_targets / T, dim=1)
        kd_loss = F.kl_div(log_probs, soft_targets_scaled, reduction='batchmean') * (T * T)
        return (1 - cfg.kd_alpha) * hard_loss + cfg.kd_alpha * kd_loss

    return hard_loss


class CosineWarmupScheduler:
    def __init__(self, optimizer, warmup_epochs, total_epochs, steps_per_epoch):
        self.optimizer = optimizer
        self.warmup_steps = warmup_epochs * steps_per_epoch
        self.total_steps = total_epochs * steps_per_epoch
        self.current_step = 0
        self.base_lrs = [pg['lr'] for pg in optimizer.param_groups]

    def step(self):
        self.current_step += 1
        if self.current_step <= self.warmup_steps:
            scale = self.current_step / self.warmup_steps
        else:
            progress = (self.current_step - self.warmup_steps) / (
                self.total_steps - self.warmup_steps
            )
            scale = 0.5 * (1 + np.cos(np.pi * progress))
        for pg, base_lr in zip(self.optimizer.param_groups, self.base_lrs):
            pg['lr'] = base_lr * scale

## Section 4: Train Loop (SAM + SWA + Progressive Resizing)

In [9]:
def train_one_phase(model, train_loader, val_loader, optimizer, scheduler,
                    cfg, phase_epochs, swa_model=None, swa_start_epoch=None,
                    ema_model=None):
    """한 phase (하나의 resolution)의 학습 루프."""
    best_val_loss = float('inf')
    best_ema_val_loss = float('inf')
    patience_counter = 0
    best_state = None
    best_ema_state = None
    history = []

    for epoch in range(phase_epochs):
        # === TRAIN ===
        model.train()
        train_losses = []
        optimizer.zero_grad()

        last_micro_batch = None

        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{phase_epochs} [Train]', leave=False)
        for step, batch in enumerate(pbar):
            front = batch['front'].to(device)
            top = batch['top'].to(device)
            labels = batch['label'].to(device)
            soft_labels = batch['soft_label'].float().to(device)

            # Mixup / CutMix
            if random.random() < cfg.mix_prob:
                if random.random() < 0.5:
                    front, top, la, lb, sa, sb, lam = mixup_data(
                        front, top, labels, soft_labels, cfg.mixup_alpha
                    )
                else:
                    front, top, la, lb, sa, sb, lam = cutmix_data(
                        front, top, labels, soft_labels, cfg.cutmix_alpha
                    )
            else:
                la, lb, sa, sb, lam = labels, labels, soft_labels, soft_labels, 1.0

            if cfg.use_sam:
                with autocast('cuda', dtype=torch.bfloat16):
                    logits = model(front, top)
                    loss = compute_loss(logits, la, lb, sa, sb, lam, cfg)
                    loss_scaled = loss / cfg.grad_accum

                loss_scaled.backward()
                last_micro_batch = (front, top, la, lb, sa, sb, lam)

                if (step + 1) % cfg.grad_accum == 0:
                    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.first_step()
                    optimizer.zero_grad()

                    f2, t2, la2, lb2, sa2, sb2, lam2 = last_micro_batch
                    with autocast('cuda', dtype=torch.bfloat16):
                        logits2 = model(f2, t2)
                        loss2 = compute_loss(logits2, la2, lb2, sa2, sb2, lam2, cfg)

                    loss2.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.second_step()
                    optimizer.zero_grad()
                    scheduler.step()

                    # EMA update (CPU에서 수행)
                    if ema_model is not None:
                        ema_model.update(model)
            else:
                with autocast('cuda', dtype=torch.bfloat16):
                    logits = model(front, top)
                    loss = compute_loss(logits, la, lb, sa, sb, lam, cfg)
                    loss_scaled = loss / cfg.grad_accum

                loss_scaled.backward()

                if (step + 1) % cfg.grad_accum == 0:
                    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()
                    optimizer.zero_grad()
                    scheduler.step()

                    # EMA update (CPU에서 수행)
                    if ema_model is not None:
                        ema_model.update(model)

            train_losses.append(loss.item())
            pbar.set_postfix({'loss': f'{np.mean(train_losses[-20:]):.4f}'})

        # SWA update
        if swa_model is not None and swa_start_epoch is not None:
            if epoch >= swa_start_epoch:
                swa_model.update_parameters(model)

        # === VALIDATE (regular model) ===
        model.eval()
        val_preds = []
        val_labels = []

        with torch.no_grad():
            for batch in val_loader:
                front = batch['front'].to(device)
                top = batch['top'].to(device)
                with autocast('cuda', dtype=torch.bfloat16):
                    logits = model(front, top)
                probs = F.softmax(logits.float(), dim=1).cpu().numpy()
                val_preds.append(probs)
                val_labels.append(batch['label'].numpy())

        val_preds = np.concatenate(val_preds)
        val_labels = np.concatenate(val_labels)
        val_logloss = log_loss(val_labels, val_preds, labels=[0, 1])
        val_auc = roc_auc_score(val_labels, val_preds[:, 1])

        # === VALIDATE (EMA model) — GPU로 잠시 올렸다가 다시 CPU로 ===
        ema_val_logloss = None
        if ema_model is not None:
            ema_model.module.to(device)
            ema_model.module.eval()
            ema_preds = []
            with torch.no_grad():
                for batch in val_loader:
                    front = batch['front'].to(device)
                    top = batch['top'].to(device)
                    with autocast('cuda', dtype=torch.bfloat16):
                        logits = ema_model.module(front, top)
                    probs = F.softmax(logits.float(), dim=1).cpu().numpy()
                    ema_preds.append(probs)
            ema_preds = np.concatenate(ema_preds)
            ema_val_logloss = log_loss(val_labels, ema_preds, labels=[0, 1])
            ema_model.module.cpu()  # 다시 CPU로
            torch.cuda.empty_cache()

        train_loss_mean = np.mean(train_losses)
        lr_current = optimizer.param_groups[0]['lr']
        log_msg = (f'Epoch {epoch+1:2d} | TrainLoss: {train_loss_mean:.4f} | '
                   f'ValLogLoss: {val_logloss:.4f} | ValAUC: {val_auc:.4f} | LR: {lr_current:.6f}')
        if ema_val_logloss is not None:
            log_msg += f' | EMA_ValLogLoss: {ema_val_logloss:.4f}'
        print(log_msg)

        history.append({
            'epoch': epoch + 1, 'train_loss': train_loss_mean,
            'val_logloss': val_logloss, 'val_auc': val_auc,
            'ema_val_logloss': ema_val_logloss,
        })

        # Early stopping (regular model)
        if val_logloss < best_val_loss:
            best_val_loss = val_logloss
            patience_counter = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            print(f'  -> Best model! LogLoss: {best_val_loss:.4f}')
        else:
            patience_counter += 1
            if patience_counter >= cfg.early_stopping_patience:
                print(f'  -> Early stopping at epoch {epoch+1}')
                break

        # Track best EMA
        if ema_val_logloss is not None and ema_val_logloss < best_ema_val_loss:
            best_ema_val_loss = ema_val_logloss
            best_ema_state = {k: v.cpu().clone() for k, v in ema_model.module.state_dict().items()}
            print(f'  -> Best EMA! LogLoss: {best_ema_val_loss:.4f}')

    return best_state, best_val_loss, val_labels, history, best_ema_state, best_ema_val_loss

## Section 5: 5-Fold CV 실행

In [10]:
def _create_optimizer_and_scheduler(model, cfg, train_loader, epochs, lr_scale=1.0):
    """Optimizer, Scheduler 생성 헬퍼."""
    backbone_params = (
        list(model.backbone_front.parameters()) +
        list(model.backbone_top.parameters())
    )
    head_params = list(model.head.parameters())
    fusion_params = list(model.fusion.parameters())
    gem_params = []
    if cfg.use_gem:
        gem_params = list(model.gem_front.parameters()) + list(model.gem_top.parameters())

    param_groups = [
        {'params': backbone_params, 'lr': cfg.lr_backbone * lr_scale},
        {'params': head_params + gem_params + fusion_params, 'lr': cfg.lr_head * lr_scale},
    ]

    if cfg.use_sam:
        optimizer = SAM(
            param_groups, base_optimizer=torch.optim.AdamW,
            rho=cfg.sam_rho, weight_decay=cfg.weight_decay
        )
    else:
        optimizer = torch.optim.AdamW(param_groups, weight_decay=cfg.weight_decay)

    steps_per_epoch = len(train_loader) // cfg.grad_accum
    scheduler = CosineWarmupScheduler(
        optimizer, cfg.warmup_epochs, epochs, steps_per_epoch
    )
    return optimizer, scheduler


def train_one_fold(fold, train_idx, val_idx, all_df, cfg):
    print(f'\n{"="*60}')
    print(f'FOLD {fold}')
    print(f'{"="*60}')

    train_data = all_df.iloc[train_idx]
    val_data = all_df.iloc[val_idx]
    print(f'Train: {len(train_data)} | Val: {len(val_data)}')
    print(f'Val unstable ratio: {val_data["label_int"].mean():.3f}')

    data_dir_path = Path(cfg.data_dir)
    best_model_path = exp_dir / f'best_fold{fold}.pt'

    # Model
    model = DualStreamModelV2(
        cfg.backbone, drop_path_rate=cfg.drop_path_rate,
        use_gem=cfg.use_gem, gem_p=cfg.gem_p_init
    ).to(device)

    # === Progressive Resizing: 2-phase ===
    if cfg.progressive_resize:
        phase1_epochs = int(cfg.epochs * cfg.phase1_pct)
        phase2_epochs = cfg.epochs - phase1_epochs
        phase1_bs = min(cfg.batch_size * 2, 32)
        phase2_bs = cfg.batch_size

        # --- Phase 1: 저해상도 ---
        print(f'\n--- Phase 1: {cfg.img_size_phase1}px, {phase1_epochs} epochs, bs={phase1_bs} ---')
        train_ds1 = StructuralDataset(train_data, data_dir_path, get_train_transforms(cfg.img_size_phase1))
        val_ds1 = StructuralDataset(val_data, data_dir_path, get_val_transforms(cfg.img_size_phase1))
        train_loader1 = DataLoader(train_ds1, batch_size=phase1_bs, shuffle=True, num_workers=0, pin_memory=True, drop_last=True)
        val_loader1 = DataLoader(val_ds1, batch_size=phase1_bs * 2, shuffle=False, num_workers=0, pin_memory=True)

        optimizer1, scheduler1 = _create_optimizer_and_scheduler(model, cfg, train_loader1, phase1_epochs)

        # EMA for phase 1
        ema_model = None
        if cfg.use_ema:
            ema_model = ModelEmaV2(model, decay=cfg.ema_decay)

        best_state_p1, best_loss_p1, _, _, best_ema_state_p1, best_ema_loss_p1 = train_one_phase(
            model, train_loader1, val_loader1, optimizer1, scheduler1,
            cfg, phase1_epochs, ema_model=ema_model
        )
        print(f'Phase 1 Best: {best_loss_p1:.4f}')
        if cfg.use_ema:
            print(f'Phase 1 EMA Best: {best_ema_loss_p1:.4f}')

        # Phase 1 best weights로 복원
        if cfg.use_ema and best_ema_loss_p1 < best_loss_p1:
            model.load_state_dict(best_ema_state_p1)
            print('  -> Using EMA weights for Phase 2 init')
        else:
            model.load_state_dict(best_state_p1)

        model.to(device)
        del train_ds1, val_ds1, train_loader1, val_loader1
        torch.cuda.empty_cache()

        # --- Phase 2: 고해상도 ---
        print(f'\n--- Phase 2: {cfg.img_size}px, {phase2_epochs} epochs, bs={phase2_bs} ---')
        train_ds2 = StructuralDataset(train_data, data_dir_path, get_train_transforms(cfg.img_size))
        val_ds2 = StructuralDataset(val_data, data_dir_path, get_val_transforms(cfg.img_size))
        train_loader2 = DataLoader(train_ds2, batch_size=phase2_bs, shuffle=True, num_workers=0, pin_memory=True, drop_last=True)
        val_loader2 = DataLoader(val_ds2, batch_size=phase2_bs * 2, shuffle=False, num_workers=0, pin_memory=True)

        optimizer2, scheduler2 = _create_optimizer_and_scheduler(model, cfg, train_loader2, phase2_epochs)

        # Recreate EMA from phase 2 init
        ema_model2 = None
        if cfg.use_ema:
            ema_model2 = ModelEmaV2(model, decay=cfg.ema_decay)

        # SWA only in Phase 2
        swa_model = None
        swa_start_epoch = None
        if cfg.use_swa:
            swa_model = AveragedModel(model)
            swa_start_epoch = int(phase2_epochs * cfg.swa_start_pct)
            print(f'SWA starts at Phase 2 epoch {swa_start_epoch + 1}')

        best_state, best_loss, val_labels, _, best_ema_state, best_ema_loss = train_one_phase(
            model, train_loader2, val_loader2, optimizer2, scheduler2,
            cfg, phase2_epochs, swa_model, swa_start_epoch, ema_model=ema_model2
        )
        val_loader = val_loader2

    else:
        # Non-progressive: single phase (original behavior)
        train_ds = StructuralDataset(train_data, data_dir_path, get_train_transforms(cfg.img_size))
        val_ds = StructuralDataset(val_data, data_dir_path, get_val_transforms(cfg.img_size))
        train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=0, pin_memory=True, drop_last=True)
        val_loader = DataLoader(val_ds, batch_size=cfg.batch_size * 2, shuffle=False, num_workers=0, pin_memory=True)

        optimizer, scheduler = _create_optimizer_and_scheduler(model, cfg, train_loader, cfg.epochs)

        ema_model2 = None
        if cfg.use_ema:
            ema_model2 = ModelEmaV2(model, decay=cfg.ema_decay)

        swa_model = None
        swa_start_epoch = None
        if cfg.use_swa:
            swa_model = AveragedModel(model)
            swa_start_epoch = int(cfg.epochs * cfg.swa_start_pct)
            print(f'SWA starts at epoch {swa_start_epoch + 1}')

        best_state, best_loss, val_labels, _, best_ema_state, best_ema_loss = train_one_phase(
            model, train_loader, val_loader, optimizer, scheduler,
            cfg, cfg.epochs, swa_model, swa_start_epoch, ema_model=ema_model2
        )

    print(f'Best LogLoss: {best_loss:.4f}')
    if cfg.use_ema:
        print(f'Best EMA LogLoss: {best_ema_loss:.4f}')

    # EMA vs regular: 더 좋은 쪽 선택
    if cfg.use_ema and best_ema_state is not None and best_ema_loss < best_loss:
        print('  -> EMA model is better! Using EMA weights.')
        best_state = best_ema_state
        best_loss = best_ema_loss

    # SWA: update batch norm and evaluate
    if swa_model is not None:
        print('Updating SWA batch norm statistics...')
        swa_train_ds = StructuralDataset(
            train_data, data_dir_path, get_val_transforms(cfg.img_size)
        )
        swa_loader = DataLoader(
            swa_train_ds, batch_size=cfg.batch_size, shuffle=True,
            num_workers=0, pin_memory=True
        )

        swa_model.to(device)
        swa_model.train()
        with torch.no_grad():
            for batch in tqdm(swa_loader, desc='SWA BN update', leave=False):
                front = batch['front'].to(device)
                top = batch['top'].to(device)
                swa_model(front, top)

        swa_model.eval()
        swa_preds = []
        with torch.no_grad():
            for batch in val_loader:
                front = batch['front'].to(device)
                top = batch['top'].to(device)
                with autocast('cuda', dtype=torch.bfloat16):
                    logits = swa_model(front, top)
                probs = F.softmax(logits.float(), dim=1).cpu().numpy()
                swa_preds.append(probs)
        swa_preds = np.concatenate(swa_preds)
        swa_logloss = log_loss(val_labels, swa_preds, labels=[0, 1])
        print(f'SWA LogLoss: {swa_logloss:.4f} (vs best: {best_loss:.4f})')

        if swa_logloss < best_loss:
            print('  -> SWA model is better! Using SWA weights.')
            best_state = {k: v.cpu().clone() for k, v in swa_model.module.state_dict().items()}
            best_loss = swa_logloss

    # Save best model
    torch.save(best_state, best_model_path)
    print(f'Best model saved: {best_model_path}')

    # Final OOF prediction
    model.load_state_dict(best_state)
    model.to(device)
    model.eval()

    oof_preds = []
    with torch.no_grad():
        for batch in val_loader:
            front = batch['front'].to(device)
            top = batch['top'].to(device)
            with autocast('cuda', dtype=torch.bfloat16):
                logits = model(front, top)
            probs = F.softmax(logits.float(), dim=1).cpu().numpy()
            oof_preds.append(probs)

    oof_preds = np.concatenate(oof_preds)
    oof_logloss = log_loss(val_labels, oof_preds, labels=[0, 1])
    print(f'\nFold {fold} Final OOF LogLoss: {oof_logloss:.4f}')

    del model, swa_model
    torch.cuda.empty_cache()

    return oof_preds, val_idx, oof_logloss

In [11]:
# 5-Fold CV 실행
skf = StratifiedKFold(n_splits=cfg.n_folds, shuffle=True, random_state=cfg.seed)

oof_predictions = np.zeros((len(all_df), 2))
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(all_df, all_df['label_int'])):
    oof_preds, val_idx_out, fold_score = train_one_fold(
        fold, train_idx, val_idx, all_df, cfg
    )
    oof_predictions[val_idx_out] = oof_preds
    fold_scores.append(fold_score)

# 전체 CV 결과
overall_logloss = log_loss(all_df['label_int'].values, oof_predictions, labels=[0, 1])
overall_auc = roc_auc_score(all_df['label_int'].values, oof_predictions[:, 1])

print(f'\n{"="*60}')
print(f'OVERALL CV RESULTS ({cfg.exp_name})')
print(f'{"="*60}')
for i, score in enumerate(fold_scores):
    print(f'  Fold {i}: LogLoss = {score:.4f}')
print(f'  Mean:   LogLoss = {np.mean(fold_scores):.4f} +/- {np.std(fold_scores):.4f}')
print(f'  Overall LogLoss = {overall_logloss:.4f}')
print(f'  Overall AUC     = {overall_auc:.4f}')

# OOF 예측 저장
np.save(exp_dir / 'oof_preds.npy', oof_predictions)
# OOF logits도 저장 (calibration용)
np.save(exp_dir / 'oof_logits.npy', np.log(np.clip(oof_predictions, 1e-15, 1)))
with open(exp_dir / 'cv_score.txt', 'w') as f:
    f.write(f'{overall_logloss:.6f}')
print(f'OOF predictions saved to {exp_dir}')


FOLD 0
Train: 880 | Val: 220
Val unstable ratio: 0.505



--- Phase 1: 224px, 12 epochs, bs=4 ---


Epoch  1 | TrainLoss: 0.7353 | ValLogLoss: 0.9231 | ValAUC: 0.8148 | LR: 0.000050 | EMA_ValLogLoss: 0.7136
  -> Best model! LogLoss: 0.9231
  -> Best EMA! LogLoss: 0.7136


Epoch  2 | TrainLoss: 0.5501 | ValLogLoss: 0.7161 | ValAUC: 0.9058 | LR: 0.000100 | EMA_ValLogLoss: 0.7134
  -> Best model! LogLoss: 0.7161
  -> Best EMA! LogLoss: 0.7134


Epoch  3 | TrainLoss: 0.5251 | ValLogLoss: 0.6575 | ValAUC: 0.9451 | LR: 0.000098 | EMA_ValLogLoss: 0.7125
  -> Best model! LogLoss: 0.6575
  -> Best EMA! LogLoss: 0.7125


...

Epoch  7 | TrainLoss: 0.5495 | ValLogLoss: 0.6980 | ValAUC: 0.3739 | LR: 0.000078 | EMA_ValLogLoss: 0.5476


Epoch  8 | TrainLoss: 0.5352 | ValLogLoss: 0.7204 | ValAUC: 0.4379 | LR: 0.000069 | EMA_ValLogLoss: 0.5486


Epoch  9 | TrainLoss: 0.5802 | ValLogLoss: 0.8145 | ValAUC: 0.4829 | LR: 0.000060 | EMA_ValLogLoss: 0.5486


Epoch 10 | TrainLoss: 0.5506 | ValLogLoss: 0.7003 | ValAUC: 0.4644 | LR: 0.000050 | EMA_ValLogLoss: 0.5488
  -> Early stopping at epoch 10
Best LogLoss: 0.6645
Best EMA LogLoss: 0.5335
  -> EMA model is better! Using EMA weights.
Updating SWA batch norm statistics...


SWA LogLoss: 0.5327 (vs best: 0.5335)
  -> SWA model is better! Using SWA weights.
Best model saved: ..\outputs\v2_convnextv2_base\best_fold4.pt

Fold 4 Final OOF LogLoss: 0.5327

OVERALL CV RESULTS (v2_convnextv2_base)
  Fold 0: LogLoss = 0.4339
  Fold 1: LogLoss = 0.2109
  Fold 2: LogLoss = 0.4361
  Fold 3: LogLoss = 0.2292
  Fold 4: LogLoss = 0.5327
  Mean:   LogLoss = 0.3685 +/- 0.1265
  Overall LogLoss = 0.3685
  Overall AUC     = 0.9315
OOF predictions saved to ..\outputs\v2_convnextv2_base


## Section 6: Test Inference + TTA

In [12]:
def predict_test(cfg):
    """5-Fold 모델로 Test 예측 (Multi-Scale TTA: scales × flips)"""
    data_dir_path = Path(cfg.data_dir)
    test_df_local = pd.read_csv(data_dir_path / 'sample_submission.csv')
    tta_scales = cfg.tta_scales  # e.g. [384, 448, 512]

    all_preds = []

    for fold in range(cfg.n_folds):
        print(f'Predicting with fold {fold} model...')
        model_path = exp_dir / f'best_fold{fold}.pt'
        model = DualStreamModelV2(
            cfg.backbone, drop_path_rate=0.0,
            use_gem=cfg.use_gem, gem_p=cfg.gem_p_init
        ).to(device)
        model.load_state_dict(torch.load(model_path, weights_only=True))
        model.eval()

        fold_preds = []

        # Multi-scale TTA: each scale × (no flip, flip) = 2*len(scales) passes
        for scale in tta_scales:
            for flip in [False, True]:
                transforms = get_multiscale_tta_transforms(scale, flip=flip)
                test_ds = StructuralDataset(
                    test_df_local, data_dir_path, transforms, is_test=True
                )
                test_loader = DataLoader(
                    test_ds, batch_size=cfg.batch_size, shuffle=False,
                    num_workers=0, pin_memory=True
                )

                tta_preds = []
                with torch.no_grad():
                    for batch in test_loader:
                        front = batch['front'].to(device)
                        top = batch['top'].to(device)
                        with autocast('cuda', dtype=torch.bfloat16):
                            logits = model(front, top)
                        probs = F.softmax(logits.float(), dim=1).cpu().numpy()
                        tta_preds.append(probs)

                tta_preds = np.concatenate(tta_preds)
                fold_preds.append(tta_preds)

        print(f'  Fold {fold}: {len(fold_preds)} TTA passes ({len(tta_scales)} scales × 2 flips)')
        fold_mean = np.mean(fold_preds, axis=0)
        all_preds.append(fold_mean)

        del model
        torch.cuda.empty_cache()

    test_preds = np.mean(all_preds, axis=0)
    print(f'Test predictions shape: {test_preds.shape}')
    print(f'Unstable prob range: [{test_preds[:, 1].min():.4f}, {test_preds[:, 1].max():.4f}]')

    return test_preds


test_preds = predict_test(cfg)
np.save(exp_dir / 'test_preds.npy', test_preds)
print(f'Test predictions saved to {exp_dir / "test_preds.npy"}')

Predicting with fold 0 model...
  Fold 0: 6 TTA passes (3 scales × 2 flips)
Predicting with fold 1 model...
  Fold 1: 6 TTA passes (3 scales × 2 flips)
Predicting with fold 2 model...
  Fold 2: 6 TTA passes (3 scales × 2 flips)
Predicting with fold 3 model...
  Fold 3: 6 TTA passes (3 scales × 2 flips)
Predicting with fold 4 model...
  Fold 4: 6 TTA passes (3 scales × 2 flips)
Test predictions shape: (1000, 2)
Unstable prob range: [0.2166, 0.8831]
Test predictions saved to ..\outputs\v2_convnextv2_base\test_preds.npy


## Section 7: 고급 확률 보정 (Temp/Platt/Isotonic 자동 선택)

In [13]:
from scipy.optimize import minimize_scalar


def temperature_scaling(oof_probs, y_true, test_probs):
    """Temperature Scaling: 최적 T 탐색"""
    def temp_logloss(T):
        scaled = np.exp(np.log(np.clip(oof_probs, 1e-15, 1)) / T)
        scaled = scaled / scaled.sum(axis=1, keepdims=True)
        return log_loss(y_true, scaled, labels=[0, 1])

    result = minimize_scalar(temp_logloss, bounds=(0.3, 5.0), method='bounded')
    best_T = result.x
    cal_logloss = result.fun

    # Apply to test
    test_scaled = np.exp(np.log(np.clip(test_probs, 1e-15, 1)) / best_T)
    test_scaled = test_scaled / test_scaled.sum(axis=1, keepdims=True)

    print(f'  Temperature Scaling: T={best_T:.4f}, OOF LogLoss={cal_logloss:.6f}')
    return test_scaled, cal_logloss, {'method': 'temperature', 'T': best_T}


def platt_scaling(oof_probs, y_true, test_probs):
    """Platt Scaling: LogisticRegression on OOF logits"""
    oof_logits = np.log(np.clip(oof_probs[:, 1] / oof_probs[:, 0], 1e-15, 1e15))
    test_logits = np.log(np.clip(test_probs[:, 1] / test_probs[:, 0], 1e-15, 1e15))

    lr = LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000)
    lr.fit(oof_logits.reshape(-1, 1), y_true)

    oof_cal = lr.predict_proba(oof_logits.reshape(-1, 1))
    cal_logloss = log_loss(y_true, oof_cal, labels=[0, 1])

    test_cal = lr.predict_proba(test_logits.reshape(-1, 1))

    print(f'  Platt Scaling: OOF LogLoss={cal_logloss:.6f}')
    return test_cal, cal_logloss, {'method': 'platt', 'model': lr}


def isotonic_calibration(oof_probs, y_true, test_probs):
    """Isotonic Regression calibration"""
    ir = IsotonicRegression(out_of_bounds='clip')
    ir.fit(oof_probs[:, 1], y_true)

    oof_cal_unstable = ir.predict(oof_probs[:, 1])
    oof_cal = np.stack([1 - oof_cal_unstable, oof_cal_unstable], axis=1)
    cal_logloss = log_loss(y_true, oof_cal, labels=[0, 1])

    test_cal_unstable = ir.predict(test_probs[:, 1])
    test_cal = np.stack([1 - test_cal_unstable, test_cal_unstable], axis=1)

    print(f'  Isotonic Regression: OOF LogLoss={cal_logloss:.6f}')
    return test_cal, cal_logloss, {'method': 'isotonic', 'model': ir}


def auto_calibrate(oof_probs, y_true, test_probs):
    """세 가지 보정 방법 중 OOF LogLoss 최소인 방법 자동 선택"""
    print('\n=== Calibration Comparison ===')
    baseline_logloss = log_loss(y_true, oof_probs, labels=[0, 1])
    print(f'  Baseline (no calibration): OOF LogLoss={baseline_logloss:.6f}')

    results = []

    # Temperature Scaling
    test_temp, ll_temp, info_temp = temperature_scaling(oof_probs, y_true, test_probs)
    results.append(('Temperature', test_temp, ll_temp, info_temp))

    # Platt Scaling
    test_platt, ll_platt, info_platt = platt_scaling(oof_probs, y_true, test_probs)
    results.append(('Platt', test_platt, ll_platt, info_platt))

    # Isotonic Regression
    test_iso, ll_iso, info_iso = isotonic_calibration(oof_probs, y_true, test_probs)
    results.append(('Isotonic', test_iso, ll_iso, info_iso))

    # Add baseline (no calibration)
    results.append(('None', test_probs, baseline_logloss, {'method': 'none'}))

    # Select best
    best = min(results, key=lambda x: x[2])
    print(f'\n  >>> Best calibration: {best[0]} (OOF LogLoss={best[2]:.6f})')

    return best[1], best[0], best[2]


# 보정 실행
y_true = all_df['label_int'].values
calibrated_test, cal_method, cal_score = auto_calibrate(
    oof_predictions, y_true, test_preds
)
print(f'\nSelected: {cal_method} calibration')


=== Calibration Comparison ===
  Baseline (no calibration): OOF LogLoss=0.368542
  Temperature Scaling: T=0.5470, OOF LogLoss=0.325899
  Platt Scaling: OOF LogLoss=0.324700
  Isotonic Regression: OOF LogLoss=0.281415

  >>> Best calibration: Isotonic (OOF LogLoss=0.281415)

Selected: Isotonic calibration


## Section 8: 단일 모델 제출

In [14]:
data_dir_path = Path(cfg.data_dir)
test_df_sub = pd.read_csv(data_dir_path / 'sample_submission.csv')

# float64로 변환 (float16 비교 문제 방지)
unstable_prob = calibrated_test[:, 1].astype(np.float64)
unstable_prob = np.clip(unstable_prob, 1e-15, 1 - 1e-15)
stable_prob = 1.0 - unstable_prob

submission = pd.DataFrame({
    'id': test_df_sub['id'],
    'unstable_prob': unstable_prob,
    'stable_prob': stable_prob,
})

assert (submission[['unstable_prob', 'stable_prob']].sum(axis=1) - 1.0).abs().max() < 1e-10

sub_path = Path('../submissions') / f'{cfg.exp_name}_submission.csv'
sub_path.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(sub_path, encoding='UTF-8-sig', index=False)
print(f'Submission saved: {sub_path}')
print(f'Shape: {submission.shape}')
submission.head(10)

Submission saved: ..\submissions\v2_convnextv2_base_submission.csv
Shape: (1000, 3)


,id,unstable_prob,stable_prob
0,TEST_0001,0.268041,0.731959
1,TEST_0002,0.899083,0.100917
2,TEST_0003,0.951613,0.048387
3,TEST_0004,0.899083,0.100917
4,TEST_0005,0.268041,0.731959
5,TEST_0006,0.951613,0.048387
6,TEST_0007,0.268041,0.731959
7,TEST_0008,0.951613,0.048387
8,TEST_0009,0.899083,0.100917
9,TEST_0010,0.268041,0.731959


## Section 9: Pseudo-Labeling (고신뢰 test 샘플 재학습)

1라운드 모델의 예측에서 고신뢰 샘플 (prob > threshold or < 1-threshold)을 pseudo-label로 추가 후 재학습.

In [1]:
if cfg.use_pseudo:
    print('\n' + '='*60)
    print('PSEUDO-LABELING')
    print('='*60)

    # 고신뢰 샘플 선택
    test_df_pseudo = pd.read_csv(data_dir_path / 'sample_submission.csv')
    unstable_probs = calibrated_test[:, 1]

    high_conf_mask = (unstable_probs > cfg.pseudo_threshold) | (unstable_probs < (1 - cfg.pseudo_threshold))
    pseudo_df = test_df_pseudo[high_conf_mask].copy()
    pseudo_probs = unstable_probs[high_conf_mask]

    pseudo_df['label'] = np.where(pseudo_probs > 0.5, 'unstable', 'stable')
    pseudo_df['label_int'] = (pseudo_probs > 0.5).astype(int)
    pseudo_df['soft_unstable_prob'] = pseudo_probs
    pseudo_df['split'] = 'test'

    print(f'High-confidence samples: {len(pseudo_df)} / {len(test_df_pseudo)}')
    print(f'  Unstable: {(pseudo_df["label_int"] == 1).sum()}')
    print(f'  Stable: {(pseudo_df["label_int"] == 0).sum()}')

    if len(pseudo_df) > 0:
        all_df_pseudo = pd.concat([all_df, pseudo_df], ignore_index=True)
        print(f'Total samples with pseudo: {len(all_df_pseudo)}')

        pseudo_exp_dir = exp_dir / 'pseudo'
        pseudo_exp_dir.mkdir(parents=True, exist_ok=True)

        skf_pseudo = StratifiedKFold(n_splits=cfg.n_folds, shuffle=True, random_state=cfg.seed)
        oof_pseudo = np.zeros((len(all_df), 2))
        fold_scores_pseudo = []

        for fold, (train_idx, val_idx) in enumerate(skf_pseudo.split(all_df, all_df['label_int'])):
            print(f'\n--- Pseudo Fold {fold} ---')

            train_data = pd.concat([
                all_df.iloc[train_idx],
                pseudo_df
            ], ignore_index=True)
            val_data = all_df.iloc[val_idx]

            train_ds = StructuralDataset(
                train_data, data_dir_path, get_train_transforms(cfg.img_size)
            )
            val_ds = StructuralDataset(
                val_data, data_dir_path, get_val_transforms(cfg.img_size)
            )

            train_loader = DataLoader(
                train_ds, batch_size=cfg.batch_size, shuffle=True,
                num_workers=0, pin_memory=True, drop_last=True
            )
            val_loader = DataLoader(
                val_ds, batch_size=cfg.batch_size * 2, shuffle=False,
                num_workers=0, pin_memory=True
            )

            # Load best model from round 1 and fine-tune
            model = DualStreamModelV2(
                cfg.backbone, drop_path_rate=cfg.drop_path_rate,
                use_gem=cfg.use_gem, gem_p=cfg.gem_p_init
            ).to(device)
            model.load_state_dict(torch.load(
                exp_dir / f'best_fold{fold}.pt', weights_only=True
            ))

            # Optimizer with fusion params
            optimizer, scheduler = _create_optimizer_and_scheduler(
                model, cfg, train_loader, 5, lr_scale=0.1
            )

            # EMA for pseudo-labeling
            ema_model = None
            if cfg.use_ema:
                ema_model = ModelEmaV2(model, decay=cfg.ema_decay)

            best_state, best_loss, val_labels, _, best_ema_state, best_ema_loss = train_one_phase(
                model, train_loader, val_loader, optimizer, scheduler,
                cfg, 5, ema_model=ema_model
            )

            # EMA vs regular
            if cfg.use_ema and best_ema_state is not None and best_ema_loss < best_loss:
                best_state = best_ema_state
                best_loss = best_ema_loss
                print(f'  -> Using EMA weights (LogLoss: {best_ema_loss:.4f})')

            torch.save(best_state, pseudo_exp_dir / f'best_fold{fold}.pt')

            model.load_state_dict(best_state)
            model.to(device)
            model.eval()

            oof_preds = []
            with torch.no_grad():
                for batch in val_loader:
                    front = batch['front'].to(device)
                    top = batch['top'].to(device)
                    with autocast('cuda', dtype=torch.bfloat16):
                        logits = model(front, top)
                    probs = F.softmax(logits.float(), dim=1).cpu().numpy()
                    oof_preds.append(probs)

            oof_preds = np.concatenate(oof_preds)
            oof_pseudo[val_idx] = oof_preds
            fold_logloss = log_loss(val_labels, oof_preds, labels=[0, 1])
            fold_scores_pseudo.append(fold_logloss)
            print(f'Pseudo Fold {fold} LogLoss: {fold_logloss:.4f}')

            del model, ema_model
            torch.cuda.empty_cache()

        pseudo_cv = log_loss(all_df['label_int'].values, oof_pseudo, labels=[0, 1])
        print(f'\nPseudo CV LogLoss: {pseudo_cv:.4f} (vs Round 1: {overall_logloss:.4f})')

        if pseudo_cv < overall_logloss:
            print('Pseudo-labeling improved performance! Using pseudo models.')

            # Multi-scale TTA for pseudo models
            test_preds_pseudo = []
            for fold in range(cfg.n_folds):
                model = DualStreamModelV2(
                    cfg.backbone, drop_path_rate=0.0,
                    use_gem=cfg.use_gem, gem_p=cfg.gem_p_init
                ).to(device)
                model.load_state_dict(torch.load(
                    pseudo_exp_dir / f'best_fold{fold}.pt', weights_only=True
                ))
                model.eval()

                fold_preds = []
                for scale in cfg.tta_scales:
                    for flip in [False, True]:
                        transforms = get_multiscale_tta_transforms(scale, flip=flip)
                        test_ds = StructuralDataset(
                            test_df_pseudo, data_dir_path, transforms, is_test=True
                        )
                        test_loader = DataLoader(
                            test_ds, batch_size=cfg.batch_size, shuffle=False,
                            num_workers=0, pin_memory=True
                        )
                        tta_preds = []
                        with torch.no_grad():
                            for batch in test_loader:
                                front = batch['front'].to(device)
                                top = batch['top'].to(device)
                                with autocast('cuda', dtype=torch.bfloat16):
                                    logits = model(front, top)
                                probs = F.softmax(logits.float(), dim=1).cpu().numpy()
                                tta_preds.append(probs)
                        fold_preds.append(np.concatenate(tta_preds))

                test_preds_pseudo.append(np.mean(fold_preds, axis=0))
                del model
                torch.cuda.empty_cache()

            test_preds_pseudo = np.mean(test_preds_pseudo, axis=0)
            np.save(pseudo_exp_dir / 'test_preds.npy', test_preds_pseudo)
            np.save(pseudo_exp_dir / 'oof_preds.npy', oof_pseudo)
            print(f'Pseudo test predictions saved')
        else:
            print('Pseudo-labeling did not improve. Keeping round 1 models.')
else:
    print('Pseudo-labeling disabled.')

NameError: name 'cfg' is not defined

## Section 10: 멀티 Backbone 앙상블 + Stacking

모든 backbone 학습 완료 후 실행.

### 추천 backbone 목록
| exp_name | backbone | img_size_phase2 |
|----------|----------|----------|
| v2_convnextv2_base | convnextv2_base.fcmae_ft_in22k_in1k_384 | 384 |
| v2_efficientnetv2_s | efficientnetv2_rw_s.ra2_in1k | 384 |
| v2_swinv2_base | swinv2_base_window12to16_192to256.ms_in22k_ft_in1k_256 | 256 |
| v2_eva02_small | eva02_small_patch14_336.mim_in22k_ft_in1k | 336 |
| v2_maxvit_tiny | maxvit_tiny_tf_384.in1k | 384 |
| v2_caformer_s18 | caformer_s18.sail_in22k_ft_in1k_384 | 384 |

In [ ]:
from scipy.optimize import minimize

output_root = Path('../outputs')
# v2 모델만 수집
model_dirs = sorted([
    d for d in output_root.iterdir()
    if d.is_dir() and d.name.startswith('v2_') and (d / 'oof_preds.npy').exists()
])

print(f'Found {len(model_dirs)} v2 models:')
for d in model_dirs:
    cv = float(open(d / 'cv_score.txt').read().strip())
    print(f'  {d.name}: CV LogLoss = {cv:.4f}')

if len(model_dirs) < 2:
    print('\n앙상블 불가: 2개 이상 모델 필요. backbone 변경 후 재학습하세요.')
    print('위 표의 backbone 설정으로 Config만 변경 후 Run All 반복.')

In [ ]:
if len(model_dirs) >= 2:
    # OOF + Test 로딩
    oof_list = []
    test_list = []
    names = []
    for d in model_dirs:
        # pseudo가 있으면 pseudo 사용, 없으면 기본
        pseudo_dir = d / 'pseudo'
        if (pseudo_dir / 'oof_preds.npy').exists():
            oof = np.load(pseudo_dir / 'oof_preds.npy')
            test = np.load(pseudo_dir / 'test_preds.npy')
            print(f'  {d.name}: Using pseudo-labeled predictions')
        else:
            oof = np.load(d / 'oof_preds.npy')
            test = np.load(d / 'test_preds.npy')
        oof_list.append(oof)
        test_list.append(test)
        names.append(d.name)

    y_true = all_df['label_int'].values

    # === Level 1: Weighted Average Ensemble ===
    print('\n=== Level 1: Weighted Average Ensemble ===')

    # 모델 상관관계
    print('\nModel Correlation (unstable_prob):')
    corr_matrix = np.corrcoef([o[:, 1] for o in oof_list])
    for i, n1 in enumerate(names):
        for j, n2 in enumerate(names):
            if i < j:
                print(f'  {n1} vs {n2}: {corr_matrix[i,j]:.4f}')

    # 최적 가중치
    def ensemble_logloss(weights):
        weights = np.abs(weights)
        weights = weights / weights.sum()
        blended = sum(w * o for w, o in zip(weights, oof_list))
        return log_loss(y_true, blended, labels=[0, 1])

    n_models = len(oof_list)
    init_weights = np.ones(n_models) / n_models
    result = minimize(ensemble_logloss, init_weights, method='Nelder-Mead')
    best_weights = np.abs(result.x)
    best_weights = best_weights / best_weights.sum()

    print(f'\nOptimal Weights:')
    for n, w in zip(names, best_weights):
        print(f'  {n}: {w:.4f}')

    ensemble_oof = sum(w * o for w, o in zip(best_weights, oof_list))
    ens_logloss = log_loss(y_true, ensemble_oof, labels=[0, 1])
    print(f'\nWeighted Ensemble CV LogLoss: {ens_logloss:.4f}')

    # Calibrate ensemble
    ensemble_test = sum(w * t for w, t in zip(best_weights, test_list))
    cal_test_ens, cal_method_ens, cal_score_ens = auto_calibrate(
        ensemble_oof, y_true, ensemble_test
    )

    # === Level 2: Stacking (Meta-Learner) ===
    print('\n=== Level 2: Stacking Meta-Learner ===')

    # Level 1 features: 각 모델의 unstable_prob
    X_meta_oof = np.column_stack([o[:, 1] for o in oof_list])  # (1100, n_models)
    X_meta_test = np.column_stack([t[:, 1] for t in test_list])  # (1000, n_models)

    # LogisticRegression meta-learner
    print('\n--- LogisticRegression Meta-Learner ---')
    lr_meta = LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000)
    lr_meta.fit(X_meta_oof, y_true)

    lr_oof_preds = lr_meta.predict_proba(X_meta_oof)
    lr_cv = log_loss(y_true, lr_oof_preds, labels=[0, 1])
    print(f'LR Meta CV LogLoss: {lr_cv:.4f}')

    lr_test_preds = lr_meta.predict_proba(X_meta_test)

    # LightGBM meta-learner (if available)
    lgb_cv = float('inf')
    lgb_test_preds = None
    try:
        import lightgbm as lgb

        print('\n--- LightGBM Meta-Learner ---')
        # 5-fold CV for LightGBM to avoid overfitting
        lgb_oof = np.zeros(len(y_true))
        lgb_test_sum = np.zeros(len(X_meta_test))

        skf_meta = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        for mfold, (mtr, mva) in enumerate(skf_meta.split(X_meta_oof, y_true)):
            dtrain = lgb.Dataset(X_meta_oof[mtr], y_true[mtr])
            dval = lgb.Dataset(X_meta_oof[mva], y_true[mva])

            params = {
                'objective': 'binary',
                'metric': 'binary_logloss',
                'learning_rate': 0.05,
                'num_leaves': 8,
                'max_depth': 3,
                'min_data_in_leaf': 20,
                'feature_fraction': 0.8,
                'bagging_fraction': 0.8,
                'bagging_freq': 5,
                'verbose': -1,
                'seed': 42,
            }

            gbm = lgb.train(
                params, dtrain, num_boost_round=500,
                valid_sets=[dval],
                callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)],
            )
            lgb_oof[mva] = gbm.predict(X_meta_oof[mva])
            lgb_test_sum += gbm.predict(X_meta_test)

        lgb_test_preds_raw = lgb_test_sum / 5
        lgb_cv = log_loss(y_true, lgb_oof, labels=[0, 1])
        print(f'LGB Meta CV LogLoss: {lgb_cv:.4f}')

        lgb_test_preds = np.column_stack([1 - lgb_test_preds_raw, lgb_test_preds_raw])

    except ImportError:
        print('LightGBM not available, skipping.')

    # === Best Stacking Selection ===
    print('\n=== Final Model Selection ===')
    candidates = [
        ('Weighted Ensemble + Calibration', cal_test_ens, cal_score_ens),
        ('LR Stacking', lr_test_preds, lr_cv),
    ]
    if lgb_test_preds is not None:
        candidates.append(('LGB Stacking', lgb_test_preds, lgb_cv))

    for name, _, score in candidates:
        print(f'  {name}: CV LogLoss = {score:.4f}')

    best_candidate = min(candidates, key=lambda x: x[2])
    print(f'\n  >>> Best: {best_candidate[0]} (CV LogLoss = {best_candidate[2]:.4f})')

    final_test_preds = best_candidate[1]

## Section 11: 최종 제출 파일 생성

In [ ]:
if len(model_dirs) >= 2:
    test_df_sub = pd.read_csv(data_dir_path / 'sample_submission.csv')

    if final_test_preds.ndim == 1:
        final_unstable = final_test_preds.astype(np.float64)
    else:
        final_unstable = final_test_preds[:, 1].astype(np.float64)

    final_unstable = np.clip(final_unstable, 1e-15, 1 - 1e-15)
    final_stable = 1.0 - final_unstable

    final_submission = pd.DataFrame({
        'id': test_df_sub['id'],
        'unstable_prob': final_unstable,
        'stable_prob': final_stable,
    })

    assert (final_submission[['unstable_prob', 'stable_prob']].sum(axis=1) - 1.0).abs().max() < 1e-10

    final_path = Path('../submissions') / 'v2_final_submission.csv'
    final_submission.to_csv(final_path, encoding='UTF-8-sig', index=False)
    print(f'Final submission saved: {final_path}')
    print(f'Method: {best_candidate[0]}')
    print(f'CV LogLoss: {best_candidate[2]:.4f}')
    print(f'Shape: {final_submission.shape}')
    final_submission.head(10)
else:
    print('단일 모델 제출만 가능합니다. Section 8의 제출 파일을 사용하세요.')

---

## 사용 가이드

### 1차 실행 (ConvNeXt-V2-Base)
- 그대로 전체 실행 → `outputs/v2_convnextv2_base/` + 제출파일

### 2차~6차 실행 (다른 backbone)
Config 셀에서 아래만 변경 후 전체 재실행:

```python
# 2차: EfficientNetV2-S
cfg = Config(
    backbone='efficientnetv2_rw_s.ra2_in1k',
    exp_name='v2_efficientnetv2_s',
)

# 3차: SwinV2-Base
cfg = Config(
    backbone='swinv2_base_window12to16_192to256.ms_in22k_ft_in1k_256',
    exp_name='v2_swinv2_base',
    img_size_phase2=256,
)

# 4차: EVA02-Small
cfg = Config(
    backbone='eva02_small_patch14_336.mim_in22k_ft_in1k',
    exp_name='v2_eva02_small',
    img_size_phase2=336,
)

# 5차: MaxViT-Tiny
cfg = Config(
    backbone='maxvit_tiny_tf_384.in1k',
    exp_name='v2_maxvit_tiny',
)

# 6차: CAFormer-S18
cfg = Config(
    backbone='caformer_s18.sail_in22k_ft_in1k_384',
    exp_name='v2_caformer_s18',
)
```

### 앙상블 + Stacking
- 2개 이상 v2 모델이 있으면 Section 10~11 실행
- 자동으로 Weighted Average vs LR Stacking vs LGB Stacking 비교
- 최적 방법 자동 선택 → `submissions/v2_final_submission.csv`